# YOLOv3 CPU Benchmark
Measures inference speed and power on the KV260 **ARM CPU (Cortex-A53)**.

**Run from any terminal on the KV260** (SSH or Jupyter terminal)

> ⚠️ YOLOv3 is very slow on ARM CPU (~4.6 seconds per frame!). Keep N_BENCHMARK=10.
> Let the board cool down between CPU and DPU runs.

Model: `/home/ubuntu/yolov3-10.onnx` (237MB, pre-downloaded from ONNX model zoo)

In [ ]:
import os, time, numpy as np

POWER_SENSOR = "/sys/class/hwmon/hwmon2/power1_input"
N_WARMUP = 2
N_BENCHMARK = 10  # Keep low — each frame takes ~4.6 seconds!

# Smart path: check local models/ first, then /home/ubuntu/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("cpu_bench.ipynb"))
MODEL_LOCAL  = os.path.join(NOTEBOOK_DIR, "models", "yolov3-10.onnx")
MODEL_UBUNTU = "/home/ubuntu/yolov3-10.onnx"
MODEL_PATH   = MODEL_LOCAL if os.path.exists(MODEL_LOCAL) else MODEL_UBUNTU

def read_power_mw():
    with open(POWER_SENSOR) as f:
        return int(f.read().strip()) / 1000

print(f"Using model: {MODEL_PATH}")
print(f"Model exists: {os.path.exists(MODEL_PATH)}")
print(f"Current board power: {read_power_mw()/1000:.2f} W (idle)")
print(f"Estimated benchmark time: ~{N_BENCHMARK * 4.6:.0f} seconds")

## Step 1 — Load YOLOv3 ONNX Model
YOLOv3 ONNX has **two inputs**: the image AND the image shape.

In [ ]:
import onnxruntime as ort
print(f"ONNX Runtime version: {ort.__version__}")

session = ort.InferenceSession(MODEL_PATH, providers=['CPUExecutionProvider'])
print(f"Inputs:")
for inp in session.get_inputs():
    print(f"  {inp.name}: {inp.shape}")

## Step 2 — Prepare Inputs
YOLOv3 ONNX needs both the image tensor AND a shape tensor.

In [ ]:
input_image = np.random.randn(1, 3, 416, 416).astype(np.float32)
input_shape = np.array([[416, 416]], dtype=np.float32)

inputs = {
    session.get_inputs()[0].name: input_image,
    session.get_inputs()[1].name: input_shape,
}
print(f"Input image shape: {input_image.shape}")
print(f"Input shape tensor: {input_shape}")

## Step 3 — Warmup

In [ ]:
print(f"Warming up ({N_WARMUP} frames)... this will take ~{N_WARMUP*4.6:.0f}s")
for _ in range(N_WARMUP):
    session.run(None, inputs)
print("Warmup done!")

## Step 4 — Benchmark
10 frames × ~4.6s = ~46 seconds total. Board will run hot.

In [ ]:
print(f"Benchmarking ({N_BENCHMARK} frames)...")
power_samples = []
start = time.time()
for i in range(N_BENCHMARK):
    session.run(None, inputs)
    power_samples.append(read_power_mw())
    print(f"  Frame {i+1}/{N_BENCHMARK} done", flush=True)
elapsed = time.time() - start
print("Done!")

## Step 5 — Results

In [ ]:
avg_power_w = sum(power_samples) / len(power_samples) / 1000
fps = N_BENCHMARK / elapsed
latency_ms = elapsed / N_BENCHMARK * 1000
fps_per_watt = fps / avg_power_w

print("=" * 40)
print("YOLOV3 CPU BENCHMARK RESULTS")
print("=" * 40)
print(f"Platform:    KV260 ARM CPU (Cortex-A53)")
print(f"FPS:         {fps:.2f}")
print(f"Latency:     {latency_ms:.1f} ms/frame")
print(f"Power:       {avg_power_w:.2f} W")
print(f"FPS/Watt:    {fps_per_watt:.2f}")
print("=" * 40)
print()
print("Previously measured: 0.22 FPS, 4643ms, 6.25W, 0.03 FPS/Watt")